# 内部机制 3：进程边界——按引用传输与签名复核

这是内部机制支线的收尾篇。第 1 篇你看到声明如何编译成签名计划，第 2 篇你看到托管状态如何活在页 arena 里。这两篇都有一个心照不宣的前提：编译计划的那双手，和执行计划的那双手，在同一个进程里。

生产环境不是这样。vLLM 的 EngineCore 跑在**另一个进程**里——它有自己的 Python 解释器、自己的 CUDA 上下文、自己的地址空间。你的 workload 是一个活的 Python 对象：方法里有字节码，字段里可能装着函数甚至模型。把它送进子进程，最省事的办法是 `pickle.dumps(workload)`。但对一个契约框架来说，这是不可接受的：pickle 一个任意对象，等于把一段**未经签名的代码**偷偷运进 worker——worker 反序列化出什么就执行什么，driver 签过的那份数学跟它没有任何关系。

所以 PLLM 的回答是：**从不序列化函数体**。跨进程边界传输的只有三样东西——类坐标（模块 + 限定名）、模块源码的摘要、冻结字段的封闭编码。worker 进程 `import` 同一个模块、用字段值重建实例；随后 driver 要求 worker 的编译签名与自己签过的计划**逐字节相等**。于是「两个进程里跑的是不是同一份数学」从信任问题变成了校验问题。

本篇全程纯 CPU：传输机制本身不需要 GPU，我们不构造引擎，只调用 `pllm.worker` 里的 `encode_workload` / `decode_workload`——正是 EngineCore 路径上 driver 与 worker 各自执行的那两个函数。

## 线上走的到底是什么字节

先看一眼传输的真身。`encode_workload` 把一个 workload 编码成一段 JSON（`WorkloadReference`），里面没有一字节代码：

- **`module` + `qualname`**：类的坐标。worker 靠它 `importlib.import_module(module)` 再逐级 `getattr` 找回同一个类；
- **`module_digest`**：定义该类的**整个模块源码文件**的 sha256。worker import 之后重算一遍，对不上就拒绝——这是「两端代码漂移」的保险丝；
- **`fields`**：冻结 dataclass 的字段值，每个字段按三种 kind 之一封闭编码。

三种字段 kind 对应你写 workload 时会放进字段的三类东西：`value` 是封闭值（原始配置、typed contract），payload 就是值的编码；`reference` 是可导入的模块级对象（比如一个模块级函数），线上只带它的坐标，worker 端 import 同一个；`worker_transport` 是实现了 `WorkerTransportable` 协议的活对象，线上只带它自己声明的封闭 payload，到 worker 的设备上再重建。最后一种稍后细讲。

下面用内建的 `DummyWorkload` 实际编码一次——它的三个字段恰好覆盖前两种 kind。

In [1]:
import json

import torch

# pllm.worker 会 import vllm（传输层与引擎同仓），但本篇从不构造引擎。
from pllm.worker import encode_workload, decode_workload
from pllm.contracts.codec import decode_boundary, encode_boundary
from pllm.contracts.workload_reference import WorkloadReference
from pllm.workloads.dummy import DummyWorkload

# driver 端：把负载编码为 WorkloadReference，再编码为线上字节（一段 JSON）。
wire = encode_workload(DummyWorkload())
reference = decode_boundary(wire, WorkloadReference)

print(f"类坐标：{reference.module}:{reference.qualname}")
print(f"模块源码摘要（sha256）：{reference.module_digest[:16]}...（共 {len(reference.module_digest)} 个十六进制字符）")
print(f"线上字节：一段 {len(wire)} 字符的 JSON\n")
for field in reference.fields:
    where = f"{field.module}:{field.qualname}" if field.module else "（payload 内联）"
    print(f"字段 {field.name:<12} kind={field.kind:<18} 去向：{where}")

print("\n线上 JSON 全文（注意：没有任何函数字节码）：")
print(json.dumps(json.loads(wire), indent=2, ensure_ascii=False))

/workspace/pllm/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-09-11 03:04:28,256	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


类坐标：pllm.workloads.dummy:DummyWorkload
模块源码摘要（sha256）：32922e781cd2af7b...（共 64 个十六进制字符）
线上字节：一段 532 字符的 JSON

字段 config       kind=value              去向：（payload 内联）
字段 transform    kind=reference          去向：pllm.workloads.dummy:hidden_rows
字段 export_name  kind=value              去向：（payload 内联）

线上 JSON 全文（注意：没有任何函数字节码）：
{
  "fields": [
    {
      "kind": "value",
      "module": "",
      "name": "config",
      "payload": "{\"enabled\":true,\"export_every_n_steps\":256,\"export_max_cols\":16,\"export_max_rows\":1,\"layer\":0}",
      "qualname": ""
    },
    {
      "kind": "reference",
      "module": "pllm.workloads.dummy",
      "name": "transform",
      "payload": "",
      "qualname": "hidden_rows"
    },
    {
      "kind": "value",
      "module": "",
      "name": "export_name",
      "payload": "\"hidden_rows\"",
      "qualname": ""
    }
  ],
  "module": "pllm.workloads.dummy",
  "module_digest": "32922e781cd2af7be2ac7f8da06b6edd750ea073da3d715cbb49ff6a45fc606b",
  "q

输出值得逐行看。`config` 是 `DummyConfig`（一个冻结 dataclass），以 `value` 形式内联——payload 就是它的封闭编码；`transform` 是模块级函数 `pllm.workloads.dummy:hidden_rows`，线上只有坐标，payload 为空；`export_name` 是个字符串，也是 `value`。整段 JSON 里没有任何函数字节码：worker 端拿到的不是代码的拷贝，而是**找回同一份代码的地图**。

worker 端收到这段字节后做三件事，全在 `decode_workload` 里：

1. **校验摘要**：import `reference.module`，重算模块源码的 sha256，与 `module_digest` 逐字节比对，不等即拒绝；
2. **重建实例**：按字段 kind 分别还原——`value` 用声明的类型注解码回封闭值，`reference` import 坐标指向的对象，`worker_transport` 在 worker 自己的设备上调用 `transport_rebuild`——然后用这些参数调用冻结 dataclass 的构造器；
3. **重新声明**：重建出的实例在 worker 侧重新走一遍编译，产出自己的编译签名，交给 driver 复核（最后一节细说）。

你可能会问：字段里的值经过「编码再解码」会不会悄悄变样？不会——编码端在放入 `value` 字段前会现场做一次封闭往返自检，解码结果与原值不相等就直接拒绝编码，变样的值根本出不了门。

下面在同一进程里同时扮演 driver 与 worker，验证往返之后两边的**声明**逐字节一致。这里需要一份编译事实（`BackendCompileFacts`）：真实部署里它来自对 worker 后端的探针，本篇用一份纯 CPU 的探针事实代替——这和测试里构造 CPU facts 的方式相同。

In [2]:
from pllm.authoring import BackendCompileFacts
from pllm.execution.runtime_facts import model_timeline
from pllm.contracts.backend_attachment import (
    AttachmentGraphMode,
    AttachmentTopology,
    BackendPlacementProbe,
)
from pllm.contracts.state_representation import StateDeviceKind
from pllm.contracts.tensor_layout import TensorDType

# worker 端：从线上字节重建实例（device/seed 在真实路径上来自 worker 的运行时配置）。
restored = decode_workload(wire, device=torch.device("cpu"), seed=0)
print("重建实例 == 原实例：", restored == DummyWorkload())

# 一份纯 CPU 的编译事实，两端各自独立声明一次。
probe = BackendPlacementProbe(
    "tutorial.boundary@1", "a" * 64, "b" * 64, "c" * 64,
    AttachmentTopology.IN_PROCESS, AttachmentGraphMode.EAGER, StateDeviceKind.CPU,
    0, 1, 1, TensorDType.FLOAT32, 4, 8, (), ("model/tokenwise_replay@1",),
)
facts = BackendCompileFacts(
    probe,
    model_timeline(torch.device("cpu"), ("model/tokenwise_replay@1",)),
    (), 2, 0, 0, layer_count=3,
)

graph_driver, layouts_driver, attachment_driver = DummyWorkload().declare(facts)
graph_worker, layouts_worker, attachment_worker = restored.declare(facts)
print("语义图一致：", graph_driver == graph_worker,
      "| 布局请求一致：", layouts_driver == layouts_worker,
      "| 附着规格一致：", attachment_driver == attachment_worker)
print("语义图执行参数摘要：", graph_driver.execution_parameters_digest[:16], "...（两端相同）")

重建实例 == 原实例： True
语义图一致： True | 布局请求一致： True | 附着规格一致： True
语义图执行参数摘要： 810ee8cf2351ea95 ...（两端相同）


## 三条纪律，以及违反时你会看到什么

按引用传输能成立，靠的是三条纪律。它们不是文档里的忠告，而是编码器里的检查——每条违规都在传输发生之前被响亮拒绝，报错信息会点名缺哪条：

1. **类必须可导入**：写在一个 `.py` 模块里。定义在 notebook 或 `__main__` 里的类，spawn 出的 worker 无法 import——worker 进程有自己的 `__main__`，里面没有你的类；
2. **必须是冻结 dataclass**（无字段也行）：冻结保证 driver 签名之后字段不会再被改写，dataclass 保证字段清单是可枚举、可逐个编码的；
3. **字段必须可封闭传输**：封闭值、可导入的模块级引用，或 `WorkerTransportable`。lambda、闭包、活模型、文件句柄都过不去。

另外模块必须 import-safe：spawn 会在 worker 里重新 import 它，模块顶层不能有 GPU 工作。

下面把三类拒绝各触发一次。注意第三条是**worker 端**的拒绝：我们手动篡改摘要来模拟「driver 签名之后有人改了模块源码」——真实场景里它拦住的是两端代码漂移，让漂移响亮失败，而不是静默地跑不同的数学。

In [3]:
from dataclasses import dataclass, replace


@dataclass(frozen=True)
class NotebookDummy(DummyWorkload):
    """定义在本 notebook 里：__module__ 是 '__main__'，worker 无法 import。"""


try:
    encode_workload(NotebookDummy())
except ValueError as exc:
    print("[拒绝 1：类不可导入]")
    print(exc, "\n")

try:
    # transform 字段换成 lambda：闭包没有可导入的坐标。
    encode_workload(replace(DummyWorkload(), transform=lambda hidden: hidden))
except ValueError as exc:
    print("[拒绝 2：字段装着闭包/局部定义]")
    print(exc, "\n")

# 模拟「签名之后模块被改动」：篡改 module_digest，让 worker 端校验失败。
tampered = encode_boundary(replace(reference, module_digest="0" * 64))
try:
    decode_workload(tampered, device=torch.device("cpu"), seed=0)
except ValueError as exc:
    print("[拒绝 3：worker 端模块摘要不匹配]")
    print(exc)

[拒绝 1：类不可导入]
EngineCore transports workloads by reference: NotebookDummy is defined in '__main__', which a spawned worker cannot import. Move the class into an importable .py module (notebook and __main__ scopes stay in-process), or keep the default multiprocessing=False 

[拒绝 2：字段装着闭包/局部定义]
EngineCore field 'transform' holds a closure or local definition; use a module-level function the worker can import, or keep multiprocessing=False 

[拒绝 3：worker 端模块摘要不匹配]
workload module 'pllm.workloads.dummy' differs from the driver's signed copy; restart the driver from identical sources instead of editing modules between signing and spawn


## 字段里装着活模型怎么办：WorkerTransportable

读到这你可能有个疑问：训练类负载的字段里难免装着**活对象**——比如一个 HF 模型模板。封闭值编码不了它，按引用 import 的也只是类而不是这个实例。难道训练负载就走不了 EngineCore？

这就是第三种字段 kind 存在的理由。`WorkerTransportable` 是一个协议，要求活对象自己回答三个问题：

- `transport_payload()`：你的**封闭** payload 是什么？（拿不出来就当场拒绝——比如 TLM 的 `HfQvModel` 只在 meta 设备模板上给出 payload，活模型直接拒绝运输）；
- `transport_payload_type()`：payload 的静态类型，worker 用它解码，**类型不从线上读**；
- `transport_rebuild(payload, device=..., seed=...)`：在 worker 自己的设备上、用 worker 的种子重新物化——driver 的状态从不共享。

线上走的只有 payload 的封闭编码加上目标类型的坐标；worker import 该类型、检查它确实实现了协议，再调用 `transport_rebuild`。TLM 的 meta 模型模板走的就是这条路：payload 是一份 `HfQvConfig`，worker 在自己的 GPU 上按签名种子把模型重新建出来。

下面用一个测试双胆（fake transport）在同一进程里演示这条链路的两侧：driver 侧取 payload 并编码，worker 侧解码后带 `device` 与 `seed` 重建——`transport_rebuild` 里特意把种子加进结果，让你看见 worker 侧的钩子真的跑了。

In [4]:
from pllm.contracts.workload_reference import WorkerTransportable


@dataclass(frozen=True, slots=True)
class FakeTransportConfig:
    value: int


class FakeTransport:
    """WorkerTransportable 演示：活对象只运封闭 payload，到 worker 设备上重建。"""

    def __init__(self, config: FakeTransportConfig) -> None:
        self.config = config

    def transport_payload(self) -> FakeTransportConfig:
        return self.config

    @classmethod
    def transport_payload_type(cls) -> type[FakeTransportConfig]:
        return FakeTransportConfig

    @classmethod
    def transport_rebuild(cls, payload: FakeTransportConfig, *, device: torch.device, seed: int) -> "FakeTransport":
        # 真实实现会在这里按 device 与 seed 重建（如 TLM 在 worker 的 GPU 上建模型）；
        # 演示版把 seed 混进结果，证明 worker 侧钩子执行过。
        return cls(FakeTransportConfig(payload.value + seed))


live = FakeTransport(FakeTransportConfig(2))
print("通过协议检查（isinstance WorkerTransportable）：", isinstance(live, WorkerTransportable))

# driver 侧：活对象 -> 封闭 payload -> 线上字节。
payload_wire = encode_boundary(live.transport_payload())
print("线上字节只有 payload 的封闭编码：", payload_wire)

# worker 侧：按静态声明的类型解码，再在自己的设备上重建。
decoded = decode_boundary(payload_wire, FakeTransport.transport_payload_type())
rebuilt = FakeTransport.transport_rebuild(decoded, device=torch.device("cpu"), seed=7)
print("worker 重建结果：", rebuilt.config, "（= payload 值 2 + worker 种子 7）")

通过协议检查（isinstance WorkerTransportable）： True
线上字节只有 payload 的封闭编码： {"value":2}
worker 重建结果： FakeTransportConfig(value=9) （= payload 值 2 + worker 种子 7）


## 收尾：driver 的逐字节复核

传输与重建只是上半场。真正的契约闭环在安装（INSTALL）时完成：worker 用重建出的负载重新编译出自己的计划，driver 随后检查 worker 回报的 `compilation_signature` 与自己签过的计划**逐字节相等**——不等就抛出 `RuntimeError("EngineCore did not install the original signed compilation in its child")`，安装失败。重建得再像，只要编译结果有一个字节不同，这次安装就不算数。这就是「比特级一致的数学」的含义：不是相信两个进程 import 了同一个模块，而是要求两个进程交出同一个签名。

想在不匹配的代码上安装会发生什么？前面「拒绝 3」已经给过预演——摘要 drift 在 worker 端就倒下了；即使摘要侥幸相同而编译结果不同，driver 的复核也会把它拦下。两道闸，各管一段。

本篇的所有演示都在一个进程里完成，因为传输函数本身就是纯 CPU 的。端到端的证据在 GPU 测试里：`test/gpu/enginecore_smoke.py --user-only` 在真实的 EngineCore 子进程中安装一个用户 workload，由 driver 完成上述签名复核。

## 小结：两条线在此汇合

内部机制支线到此走完。回看三篇，它们回答的是同一个问题的三个剖面：声明如何变成签名计划（编译管线），签名状态如何安全地演化（页 arena 的版本、COW、pin 与预算），以及签名计划如何跨越进程边界而不失真（按引用传输 + 签名复核）。共同的心智模型只有一句话：**PLLM 从不运输代码或状态本身，只运输找回它们所需的坐标与封闭值，然后用签名证明两端拿到的是同一份数学。**

从这里你有两个方向：

- **回到算法主线**：主教程的 [04_loops](../04_loops.ipynb) 与 [05_decode_training](../05_decode_training.ipynb) 把循环与训练中观测建在图原语上——你现在知道那些原语跨过进程边界时发生了什么；
- **继续深入系统**：[doc/design/architecture.md](../../design/architecture.md) 给出整体架构，[doc/reference/authoring-contract.md](../../reference/authoring-contract.md) 逐条列出创作契约的精确边界——本篇的三条纪律在那里有正式表述。